# Validating explanations on a pediatric SFCN (public Calgary controls)

**One line.** Das et al. (2024) flagged their explainability as *preliminary* — a single coarse Grad-CAM map, with only one of four named regions tested statistically. This notebook takes that stated limitation and builds the rigorous version: a pre-registered battery that tests which explanation methods are **faithful**, **stable across methods**, and **anatomically valid** on *their exact model class*.

**Same model, proxy task.** We use the *same* network they used — the pretrained SFCN (Peng et al., 2021), frozen, six convolutional blocks kept as a feature extractor. The only differences are the small head on top and the label it predicts:

| | Das et al. (2024) | This study |
|---|---|---|
| Backbone | Pretrained SFCN, frozen | **Same** pretrained SFCN, frozen |
| Head | small dense head → PAE | small dense head → **age group** (older vs younger, median split) |
| Data | private PAE + public controls | **public controls only** (96 children, one scan each) |

Swapping the head's task does **not** invalidate the XAI. Explanation *validity* — faithfulness, the randomized-weight sanity check, cross-method agreement, resolution — is a property of the **method and the architecture**, not of the task. Testing it on a controllable proxy is standard practice (Adebayo et al. 2018; Siegel et al. 2025).

**What this does and does not show.**

| Claim | This notebook |
|---|---|
| Which XAI methods are trustworthy on the SFCN | **Yes — the deliverable** |
| Grad-CAM over-resolves / methods disagree / which fail sanity | **Yes — the headline** |
| Anything about alcohol exposure, FASD, PAE | **No — needs the private cohort** |
| Reproducing the paper's 85–88% accuracy | **No — that was the PAE task** |

The second column is the **next stage** and the reason to collaborate — not a gap in this work.


## Before you run — first-run checklist

This notebook is built to the pre-registered plan and is internally consistent, but three things depend on the live Kaggle environment and must be eyeballed on the **first** run (they are marked `# FIRST-RUN CHECK` in the code):

1. **OSF file tree** — the downloader crawls OSF project `axz5r` and filters T1 NIfTI files + a demographics table. Confirm it found scans *and* sex/age labels; the exact folder names on OSF may differ.
2. **SFCN `state_dict` keys** — after loading the pretrained checkpoint, confirm every `feature_extractor.*` key matched (the loader raises if not).
3. **Registration quality** — open the saved QC mid-slices for 2–3 scans after preprocessing. Garbage registration ⇒ garbage explanations (plan §7, Checkpoint 1).

**Compute — run the ladder in order.** The notebook ships on `MODE="smoke"`: **3–5 scans** with every slow knob (bootstrap, permutations, epochs, IG/SmoothGrad samples, masking nulls) shrunk, so the whole thing runs end to end in minutes. Its numbers are a **plumbing check, not a finding** — use it only to confirm nothing is broken. Then:

1. `MODE="smoke"` — does it run at all? (minutes)
2. `MODE="pilot"` — 20 scans, real settings. (plan Day 1)
3. `MODE="full"` — all 96 children, the deliverable. (~5–15 GPU-h, inside Kaggle's ~30 GPU-h/week)

Preprocessing runs on **CPU** (ANTs) so it does not burn GPU quota; GPU is used only for feature extraction + attributions.

**Nothing is destructive and every heavy stage caches to disk**, so a re-run resumes instead of repeating.


In [ ]:
# === Configuration (pre-registered values; changing one goes in the decision log) ===
from pathlib import Path

CFG = dict(
    MODE="full",              # "smoke" (3-5 scans, fast) | "pilot" (20) | "full" (96)
    N_SMOKE=3,                 # smoke test: just enough to prove the pipeline runs
    N_PILOT=20,
    ONE_SCAN_PER_CHILD=True,   # main analysis: one scan per child (avoids leakage)
    SEED=0,

    # --- data ---
    OSF_NODE="5kz2p",          # fork holding the files (original axz5r storage is empty)
    T1_FOLDER="T1_Dataset",    # T1_Dataset/<childID>/<sessionID>/<sessionID>.nii.gz
    PROXY_LABEL="age",         # proxy = age dichotomized at the cohort median (older=1)

    # --- model (verified source) ---
    SFCN_WEIGHTS_URL=(         # Peng et al. 2021 brain-age model; feature extractor reused
        "https://github.com/ha-ha-ha-han/UKBiobank_deep_pretrain/raw/master/"
        "brain_age/run_20190719_00_epoch_best_mae.p"
    ),
    SFCN_INPUT=(160, 192, 160),   # 1 mm MNI crop; yields a 64 x 5 x 6 x 5 feature map

    # --- attribution hyperparameters (capped per plan §8.1; logged) ---
    IG_STEPS=32,
    SG_SAMPLES=20,
    SG_STDEV=0.1,
    REX_GRID=(5, 6, 5),        # matches the SFCN feature-map grid
    LRP_RULE="EpsilonPlusFlat",

    # --- statistics (fixed now) ---
    N_BOOT=10000,
    N_PERM=10000,
    FDR_Q=0.05,
    MASK_FILLS={"mean": None, "zero": 0.0},  # second value checks for mask artifacts

    # --- CV ---
    N_FOLDS=5,
    N_REPEATS=10,              # repeated stratified-grouped CV: shrink estimate variance
    N_PERM_ACC=1000,           # permutation test that the proxy beats chance
    HEAD_EPOCHS=100,
    HEAD_LR=1e-3,
)

# Number of random null regions for the masking test (full stats vs fast smoke).
CFG["N_NULL"] = 200

# SMOKE TEST: shrink every expensive knob so the whole notebook runs in minutes.
# This proves the pipeline works end to end; the numbers are NOT interpretable.
if CFG["MODE"] == "smoke":
    CFG.update(
        N_BOOT=200, N_PERM=200, N_NULL=15, N_FOLDS=2, N_REPEATS=2, N_PERM_ACC=20, HEAD_EPOCHS=15,
        IG_STEPS=8, SG_SAMPLES=4, SMOKE_MAX_REGIONS=6,
        SMOKE_SYNTHETIC=False,  # True = fabricate volumes; skip data entirely
        SMOKE_LIGHT=False,      # True = resize-only; False = REAL ANTs prep + real atlas
                                # (heavy smoke: skull-strip + MNI registration + Harvard-Oxford)
    )
    print("=== SMOKE TEST MODE ===  results are a plumbing check, not a finding.")

# Kaggle gives a writable /kaggle/working; fall back to ./work locally.
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
DIRS = {k: ROOT / k for k in ["raw", "preproc", "features", "explanations", "results", "qc"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("MODE =", CFG["MODE"], "| workspace =", ROOT)


In [ ]:
# === Dependencies (quiet; skip if already present) ===
import importlib, subprocess, sys

def _ensure(pkgs):
    for mod, pip_name in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

deps = [("torch", "torch"), ("captum", "captum"), ("zennit", "zennit"),
        ("nibabel", "nibabel"), ("scipy", "scipy"), ("sklearn", "scikit-learn")]
_HEAVY = not CFG.get("SMOKE_SYNTHETIC") and not CFG.get("SMOKE_LIGHT")
if _HEAVY:
    # Real registration path (pilot/full): heavy, slow, flaky-on-install.
    deps += [("ants", "antspyx"), ("antspynet", "antspynet"), ("nilearn", "nilearn")]
_ensure(deps)
print("dependencies ready |", "heavy(ANTs)" if _HEAVY else
      ("synthetic" if CFG.get("SMOKE_SYNTHETIC") else "light(resize-only)"))


In [ ]:
# === Stage bundled model / template / atlas caches (heavy path only) ===
# External hosts block Kaggle datacenter IPs: antspynet's weights+templates (figshare)
# return HTTP 403, and nilearn's Harvard-Oxford mirror is unreliable too. So they are
# shipped in the attached `antsxnet-weights` dataset under opaque *.dat names (Kaggle
# auto-decompresses bare .nii.gz) and copied here into the exact cache paths each
# library checks with os.path.exists BEFORE it would attempt a download.
import shutil
from pathlib import Path

NILEARN_DATA = None   # None -> nilearn default (used only off the heavy path)
if _HEAVY:
    BUNDLE = None
    inp = Path("/kaggle/input")
    if inp.exists():
        hits = list(inp.glob("**/brainExtractionRobustT1.h5"))
        BUNDLE = hits[0].parent if hits else None
    if BUNDLE is None:
        raise FileNotFoundError(
            "Heavy path needs the 'antsxnet-weights' dataset attached "
            "(figshare/nilearn are blocked from Kaggle IPs).")

    # antspynet cache: filenames must match exactly what get_*_data() looks up.
    axn = Path(ROOT) / "antsxnet_cache"; axn.mkdir(parents=True, exist_ok=True)
    for src, dst in {
        "brainExtractionRobustT1.h5": "brainExtractionRobustT1.h5",
        "S_template3.niigz.dat":      "S_template3.nii.gz",
        "mni152.niigz.dat":           "mni152.nii.gz",
    }.items():
        d = axn / dst
        if not d.exists():
            shutil.copy(BUNDLE / src, d)
    from antspynet.utilities.get_antsxnet_data import set_antsxnet_cache_directory
    set_antsxnet_cache_directory(str(axn))

    # nilearn Harvard-Oxford: recreate the tree fetch_atlas_harvard_oxford() checks.
    NILEARN_DATA = str(Path(ROOT) / "nilearn_data")
    ho_dir = Path(NILEARN_DATA) / "fsl" / "data" / "atlases" / "HarvardOxford"
    ho_dir.mkdir(parents=True, exist_ok=True)
    for src, d in {
        "ho_cort_thr25_1mm.niigz.dat": ho_dir / "HarvardOxford-cort-maxprob-thr25-1mm.nii.gz",
        "HarvardOxford-Cortical.xml":  ho_dir.parent / "HarvardOxford-Cortical.xml",
    }.items():
        if not Path(d).exists():
            shutil.copy(BUNDLE / src, d)
    print("staged antsxnet cache ->", axn)
    print("staged nilearn data   ->", NILEARN_DATA)


In [ ]:
# === Imports, determinism, device ===
import json, os, time, copy, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## 1. Data — public Calgary Preschool T1 (OSF fork `5kz2p`)

Controls only (no PAE), with **age and sex** labels. The files live on the OSF fork `5kz2p` (the original `axz5r` storage is empty): `T1_Dataset/<PreschoolID>/<ScanID>/<ScanID>.nii.gz`, with labels in `Calgary_Preschool_Dataset_Updated_*.xlsx` (sex coded Female=0, Male=1). We build a manifest (`child_id, scan_id, age, sex, path`) and, for the main analysis, keep **one scan per child** to avoid leakage. CC-BY data — confirm the license before publishing.


In [ ]:
# === Download real scans from OSF (fork 5kz2p) + build the manifest ===
# Structure: T1_Dataset/<PreschoolID>/<ScanID>/<ScanID>.nii.gz
# Labels: Calgary_Preschool_Dataset_Updated_*.xlsx (sheet 0): PreschoolID, ScanID,
#         "Age (Years)", "Biological Sex (Female = 0; Male = 1)".
OSF_API = "https://api.osf.io/v2"

def _osf_get(url):
    req = urllib.request.Request(url, headers={"Accept": "application/vnd.api+json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)

def _root_listing(node):
    out, url = [], f"{OSF_API}/nodes/{node}/files/osfstorage/"
    while url:
        page = _osf_get(url)
        out += page["data"]
        url = page["links"].get("next")
    return out

def _folder_href(items, name):
    for it in items:
        a = it["attributes"]
        if a["kind"] == "folder" and a["name"] == name:
            return it["relationships"]["files"]["links"]["related"]["href"]
    raise FileNotFoundError(f"folder '{name}' not found on OSF node")

def osf_walk(href, parents=()):
    """Yield (parents_tuple, name, download_url) for files beneath a folder href."""
    url = href
    while url:
        page = _osf_get(url)
        for it in page["data"]:
            a = it["attributes"]
            if a["kind"] == "folder":
                sub = it["relationships"]["files"]["links"]["related"]["href"]
                yield from osf_walk(sub, parents + (a["name"],))
            else:
                yield parents, a["name"], it["links"]["download"]
        url = page["links"].get("next")

def download_scans():
    # FIRST-RUN CHECK: confirm it finds NIfTI scans under T1_Dataset and the xlsx labels.
    manifest_csv = DIRS["raw"] / "manifest.csv"
    if manifest_csv.exists():
        return pd.read_csv(manifest_csv, dtype={"child_id": str, "scan_id": str})

    root = _root_listing(CFG["OSF_NODE"])
    t1 = _folder_href(root, CFG["T1_FOLDER"])
    n_target = {"smoke": CFG["N_SMOKE"], "pilot": CFG["N_PILOT"]}.get(CFG["MODE"])

    rows, seen = [], set()
    for parents, name, dl in osf_walk(t1):
        if not name.lower().endswith((".nii", ".nii.gz")):
            continue
        child = parents[0] if parents else name.split("_")[0]   # <PreschoolID>
        scan = name.replace(".nii.gz", "").replace(".nii", "")  # <ScanID>
        if CFG["ONE_SCAN_PER_CHILD"] and child in seen:
            continue
        seen.add(child)
        out = DIRS["raw"] / f"{scan}.nii.gz"
        if not out.exists():
            urllib.request.urlretrieve(dl, out)
        rows.append(dict(child_id=child, scan_id=scan, path=str(out)))
        if n_target and len(rows) >= n_target:
            break
    man = pd.DataFrame(rows)

    # Demographics: newest *dated* Calgary_Preschool_Dataset_Updated_YYYYMMDD.xlsx
    # (plain "_v1" has no date and must not win the sort).
    import re
    cands = [it for it in root
             if it["attributes"]["name"].startswith("Calgary_Preschool_Dataset")
             and it["attributes"]["name"].endswith(".xlsx")]
    def _date(it):
        m = re.search(r"(\d{8})", it["attributes"]["name"])
        return int(m.group(1)) if m else -1
    chosen = max(cands, key=_date)
    print("demographics file:", chosen["attributes"]["name"])
    demo_path = DIRS["raw"] / "demographics.xlsx"
    if not demo_path.exists():
        urllib.request.urlretrieve(chosen["links"]["download"], demo_path)
    demo = pd.read_excel(demo_path, sheet_name=0).rename(columns={
        "PreschoolID": "child_id", "ScanID": "scan_id",
        "Age (Years)": "age", "Biological Sex (Female = 0; Male = 1)": "sex"})
    demo["scan_id"] = demo["scan_id"].astype(str).str.strip()
    man["scan_id"] = man["scan_id"].astype(str).str.strip()
    man = man.merge(demo[["scan_id", "age", "sex"]], on="scan_id", how="left").reset_index(drop=True)
    man.to_csv(manifest_csv, index=False)
    return man

def make_synthetic_manifest():
    """Smoke test: a manifest of fake scans (no OSF). Volumes are written in the next cell."""
    rng = np.random.default_rng(CFG["SEED"])
    rows = [dict(child_id=f"c{i:02d}", scan_id=f"synth{i:02d}", path="",
                 age=float(rng.integers(24, 96)), sex=["M", "F"][i % 2])
            for i in range(CFG["N_SMOKE"])]
    man = pd.DataFrame(rows)
    man.to_csv(DIRS["raw"] / "manifest.csv", index=False)
    return man

def find_kaggle_manifest():
    """Prefer an attached Kaggle dataset (robust) over in-kernel OSF download."""
    base = Path("/kaggle/input")
    if base.exists():
        for m in base.rglob("manifest.csv"):
            return m
    return None

if CFG.get("SMOKE_SYNTHETIC"):
    print("SMOKE/SYNTHETIC: fabricating", CFG["N_SMOKE"], "fake brains (no data, plumbing check only).")
    manifest = make_synthetic_manifest()
else:
    km = find_kaggle_manifest()
    if km is not None:
        manifest = pd.read_csv(km, dtype={"child_id": str, "scan_id": str})
        def _resolve(scan_id):
            # Kaggle may have decompressed .nii.gz -> .nii; match by stem, any ext.
            for ext in (".nii.gz", ".nii"):
                p = km.parent / f"{scan_id}{ext}"
                if p.exists():
                    return str(p)
            hits = [h for h in km.parent.rglob(f"{scan_id}*")
                    if h.suffix in (".nii", ".gz")]
            return str(hits[0]) if hits else ""
        manifest["path"] = manifest["scan_id"].apply(_resolve)
        print(f"loaded {len(manifest)} scans from attached Kaggle dataset: {km.parent}")
        print(manifest[["scan_id", "path"]].to_string(index=False))
    else:
        print("no Kaggle dataset attached — falling back to OSF download (local dev).")
        manifest = download_scans()

# Encode sex to {0,1}; FIRST-RUN CHECK the mapping matches the demographics coding.
if manifest["sex"].dtype == object:
    manifest["sex_label"] = (
        manifest["sex"].astype(str).str.upper().str[0].map({"M": 0, "F": 1})
    )
else:
    manifest["sex_label"] = manifest["sex"].astype("Int64")
print(manifest[["child_id", "scan_id", "age", "sex", "sex_label"]].head())
print("scans:", len(manifest), "| children:", manifest["child_id"].nunique())


## 2. Preprocessing (CPU, ANTs) — brain extraction, N4, affine to MNI, crop to 160×192×160

Matches the SFCN pretraining geometry (1 mm MNI). **Checkpoint 1 / stop rule:** if more than 25% of scans fail, stop and report (plan §4.0). QC mid-slices are saved for eyeballing.


In [ ]:
# === Preprocess each scan to the SFCN input grid ===
# ANTs/antspynet are imported lazily inside the functions so the synthetic smoke
# path does not require them at all.

def center_crop_or_pad(arr, target):
    out = np.zeros(target, dtype=arr.dtype)
    sl_in, sl_out = [], []
    for a, t in zip(arr.shape, target):
        if a >= t:
            s = (a - t) // 2; sl_in.append(slice(s, s + t)); sl_out.append(slice(0, t))
        else:
            s = (t - a) // 2; sl_in.append(slice(0, a)); sl_out.append(slice(s, s + a))
    out[tuple(sl_out)] = arr[tuple(sl_in)]
    return out

def normalize(arr):
    m = arr[arr > 0].mean() if (arr > 0).any() else 1.0
    return (arr / m).astype(np.float32)

_TEMPLATE = None
def mni_template():
    global _TEMPLATE
    if _TEMPLATE is None:
        import ants
        from antspynet.utilities import get_antsxnet_data
        _TEMPLATE = ants.image_read(get_antsxnet_data("mni152"))  # 1 mm MNI152
    return _TEMPLATE

def preprocess_one(path):
    import ants
    from antspynet.utilities import brain_extraction
    img = ants.image_read(path)
    img = ants.n4_bias_field_correction(img)
    prob = brain_extraction(img, modality="t1")           # FIRST-RUN CHECK: brain mask sane
    brain = img * ants.threshold_image(prob, 0.5, 1.0)
    reg = ants.registration(mni_template(), brain, type_of_transform="Affine")
    vol = reg["warpedmovout"].numpy()
    vol = center_crop_or_pad(vol, CFG["SFCN_INPUT"])
    return normalize(vol)

def run_preprocessing(manifest):
    fails = []
    for _, row in manifest.iterrows():
        out = DIRS["preproc"] / f"{row['scan_id']}.npy"
        if out.exists():
            continue
        try:
            vol = preprocess_one(row["path"])
            np.save(out, vol)
            mid = vol[vol.shape[0] // 2]
            plt.imsave(DIRS["qc"] / f"{row['scan_id']}.png", mid.T, cmap="gray", origin="lower")
        except Exception as e:  # noqa: BLE001
            fails.append((row["scan_id"], repr(e)))
    frac = len(fails) / max(len(manifest), 1)
    print(f"preprocessed {len(manifest) - len(fails)}/{len(manifest)} | failures: {len(fails)}")
    if fails:
        print("failures:", fails)
    # Stop rule applies to the real study, not the tiny smoke test.
    if frac > 0.25 and CFG["MODE"] != "smoke":
        raise RuntimeError(f"STOP RULE: {frac:.0%} of scans failed preprocessing. {fails}")
    return fails

if CFG.get("SMOKE_SYNTHETIC"):
    rng = np.random.default_rng(CFG["SEED"])
    for _, row in manifest.iterrows():
        out = DIRS["preproc"] / f"{row['scan_id']}.npy"
        if not out.exists():
            # Faint structure so attributions are not perfectly uniform noise.
            vol = (rng.random(CFG["SFCN_INPUT"], dtype=np.float32) * 0.3)
            vol[40:120, 48:144, 40:120] += 1.0
            np.save(out, normalize(vol))
    fails = []
    print("SMOKE/SYNTHETIC: wrote", len(manifest), "random volumes of shape", CFG["SFCN_INPUT"])
elif CFG.get("SMOKE_LIGHT"):
    # Real scans, light prep: load NIfTI, resize to the SFCN grid, normalize.
    # No skull-strip / MNI registration — those are validated in the pilot (heavy path).
    import nibabel as nib
    from scipy.ndimage import zoom
    fails = []
    for _, row in manifest.iterrows():
        out = DIRS["preproc"] / f"{row['scan_id']}.npy"
        if out.exists():
            continue
        try:
            vol = np.asarray(nib.load(row["path"]).get_fdata(), dtype=np.float32)
            while vol.ndim > 3:
                vol = vol[..., 0]
            factors = [t / s for t, s in zip(CFG["SFCN_INPUT"], vol.shape)]
            vol = zoom(vol, factors, order=1)
            vol = center_crop_or_pad(vol, CFG["SFCN_INPUT"])
            np.save(out, normalize(vol))
            plt.imsave(DIRS["qc"] / f"{row['scan_id']}.png",
                       vol[vol.shape[0] // 2].T, cmap="gray", origin="lower")
        except Exception as e:  # noqa: BLE001
            fails.append((row["scan_id"], repr(e)))
    print(f"SMOKE/LIGHT: prepped {len(manifest) - len(fails)}/{len(manifest)} real scans "
          f"to {CFG['SFCN_INPUT']} (resize+normalize).")
    if fails:
        print("failures:", fails)
else:
    fails = run_preprocessing(manifest)
manifest = manifest[~manifest["scan_id"].isin([f[0] for f in fails])].reset_index(drop=True)


## 3. Model — the original SFCN, frozen (Peng et al., 2021)

We download the released brain-age checkpoint and keep **only the six convolutional blocks** as a frozen feature extractor — exactly what Das et al. did. The classifier is discarded. The head we train is separate (§5).


In [ ]:
# === SFCN architecture + frozen feature extractor + age-group head (inlined) ===
class SFCN(nn.Module):
    """Peng et al. (2021) SFCN; only feature_extractor is reused here."""
    def __init__(self, channel_number=(32, 64, 128, 256, 256, 64), output_dim=40, dropout=True):
        super().__init__()
        n = len(channel_number)
        self.feature_extractor = nn.Sequential()
        for i in range(n):
            ic = 1 if i == 0 else channel_number[i - 1]
            oc = channel_number[i]
            mp = i < n - 1
            ks, pad = (3, 1) if mp else (1, 0)
            self.feature_extractor.add_module(f"conv_{i}", self._block(ic, oc, mp, ks, pad))
        self.classifier = nn.Sequential()
        self.classifier.add_module("average_pool", nn.AvgPool3d((5, 6, 5)))
        if dropout:
            self.classifier.add_module("dropout", nn.Dropout(0.5))
        self.classifier.add_module(f"conv_{n}", nn.Conv3d(channel_number[-1], output_dim, 1))

    @staticmethod
    def _block(ic, oc, maxpool, ks, pad):
        layers = [nn.Conv3d(ic, oc, kernel_size=ks, padding=pad), nn.BatchNorm3d(oc)]
        if maxpool:
            layers.append(nn.MaxPool3d(2, stride=2))
        layers.append(nn.ReLU())
        return nn.Sequential(*layers)

    def forward(self, x):
        return F.log_softmax(self.classifier(self.feature_extractor(x)), dim=1)


class AgeSFCN(nn.Module):
    """Frozen feature extractor + 512-unit age-group head (2-logit softmax; see decision log)."""
    def __init__(self, feature_extractor, n_features=64 * 5 * 6 * 5, hidden=512, n_classes=2):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.head(self.feature_extractor(x))


def load_feature_extractor():
    ckpt = DIRS["raw"] / "sfcn_brainage.p"
    if not ckpt.exists():
        urllib.request.urlretrieve(CFG["SFCN_WEIGHTS_URL"], ckpt)
    model = SFCN()
    state = torch.load(ckpt, map_location="cpu")
    state = state.get("state_dict", state)
    state = {k.replace("module.", ""): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    fe_missing = [k for k in missing if k.startswith("feature_extractor.")]
    # FIRST-RUN CHECK: fe_missing must be empty, or the checkpoint source is wrong (stop rule).
    assert not fe_missing, f"feature-extractor keys unmatched: {fe_missing}"
    for p in model.feature_extractor.parameters():
        p.requires_grad_(False)
    model.feature_extractor.to(DEVICE).eval()
    print("feature extractor loaded | unmatched (classifier, expected):", len(unexpected))
    return model.feature_extractor

feature_extractor = load_feature_extractor()

# Confirm geometry: 160x192x160 -> 64 x 5 x 6 x 5
with torch.no_grad():
    dummy = torch.zeros(1, 1, *CFG["SFCN_INPUT"], device=DEVICE)
    fmap = feature_extractor(dummy)
print("feature-map shape:", tuple(fmap.shape))
assert tuple(fmap.shape[1:]) == (64, 5, 6, 5), "geometry mismatch — check SFCN_INPUT"


## 4. Feature extraction (frozen forward pass)

Cache the flattened 64×5×6×5 feature vector per scan. This is the only expensive GPU step besides attributions, and it runs once.


In [ ]:
# === Extract and cache frozen features ===
def load_volume(scan_id):
    vol = np.load(DIRS["preproc"] / f"{scan_id}.npy").astype(np.float32)
    return torch.from_numpy(vol)[None, None]

@torch.no_grad()
def extract_features(manifest):
    feats = {}
    for _, row in manifest.iterrows():
        cache = DIRS["features"] / f"{row['scan_id']}.npy"
        if cache.exists():
            feats[row["scan_id"]] = np.load(cache); continue
        x = load_volume(row["scan_id"]).to(DEVICE)
        f = feature_extractor(x).flatten().cpu().numpy().astype(np.float32)
        np.save(cache, f); feats[row["scan_id"]] = f
    return feats

manifest = manifest.dropna(subset=["age"]).reset_index(drop=True)
features = extract_features(manifest)
X = np.stack([features[s] for s in manifest["scan_id"]])
_median = float(manifest["age"].median())
manifest["proxy_label"] = (manifest["age"] > _median).astype(int)  # older=1, younger=0
y = manifest["proxy_label"].to_numpy()
groups = manifest["child_id"].to_numpy()
print(f"feature matrix: {X.shape} | age median {_median:.2f} y | age-group balance:", np.bincount(y))


In [ ]:
# === PREP-ONLY: write manifest + list cached outputs, then stop (no model/XAI) ===
import shutil
manifest.to_csv(DIRS["preproc"] / "manifest_prepared.csv", index=False)
# Mirror caches to a flat, easy-to-download folder at the workspace root.
dl = ROOT / "prep_output"; (dl / "preproc").mkdir(parents=True, exist_ok=True)
(dl / "features").mkdir(parents=True, exist_ok=True)
for p in DIRS["preproc"].glob("*.npy"): shutil.copy(p, dl / "preproc" / p.name)
for p in DIRS["features"].glob("*.npy"): shutil.copy(p, dl / "features" / p.name)
manifest.to_csv(dl / "manifest.csv", index=False)
npre = len(list((dl / "preproc").glob("*.npy")))
nfeat = len(list((dl / "features").glob("*.npy")))
print(f"PREP-ONLY DONE | preprocessed arrays: {npre} | feature arrays: {nfeat}")
print("download these from the kernel output:", dl)
